# Library 00: settings and small helpers

We keep every setting of the project in this one notebook: where the files live, which dates and formats we
study, and the random seed. If we ever want to change the study window, this is the only place to touch.

It also has two helpers that all stages use:
* a logger, so every stage leaves a log file in `outputs/logs/`
* functions to save and load our CSV tables with the same date format every time

This notebook only defines things. The stage notebooks load it with `%run ./library/lib_00_settings.ipynb`.

In [ ]:
# Imports used across the project. We load them once here so the other notebooks stay short.
import json
import hashlib
import logging
import statistics
import zipfile
from dataclasses import asdict, dataclass, field
from datetime import date, datetime, timezone
from pathlib import Path
from typing import Any, Iterable

import networkx as nx
import numpy as np
import pandas as pd

## Folder paths

We don't want the notebooks to depend on where Jupyter was started from, so we search upwards from the
current folder until we find `run_pipeline.sh`. That folder is the project root and all paths start from it.

In [ ]:
def find_project_root(start_folder: Path) -> Path:
    """Go up from start_folder until we reach the folder that has run_pipeline.sh in it."""
    for folder in [start_folder, *start_folder.parents]:
        if (folder / "run_pipeline.sh").exists():
            return folder
    raise FileNotFoundError("Could not find the project root (the folder with run_pipeline.sh).")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())

# Raw data. We never edit the zip file; the unzipped folder is just a working copy we can rebuild any time.
RAW_DATA_FOLDER = PROJECT_ROOT / "data" / "raw"
RAW_ARCHIVE_PATH = RAW_DATA_FOLDER / "england_male_json.zip"
RAW_MATCH_FOLDER = RAW_DATA_FOLDER / "england_male_json"
ARCHIVE_SOURCE_URL = "https://cricsheet.org/downloads/england_male_json.zip"

# Everything we produce.
INTERIM_FOLDER = PROJECT_ROOT / "data" / "interim"        # file list and filter decisions
PROCESSED_FOLDER = PROJECT_ROOT / "data" / "processed"    # clean tables that later stages read
TABLES_FOLDER = PROJECT_ROOT / "outputs" / "tables"       # measures and check results
FIGURES_FOLDER = PROJECT_ROOT / "outputs" / "figures"     # pictures used in the report
LOGS_FOLDER = PROJECT_ROOT / "outputs" / "logs"           # one log per stage
REPORT_FOLDER = PROJECT_ROOT / "report"
TEMPLATE_PATH = PROJECT_ROOT / "docs" / "reference" / "Network_Analysis_Project_Template.docx"

## What we study

These four settings decide which matches are in the study.

In [ ]:
TEAM_NAME = "England"
TEAM_GENDER = "male"
TEAM_TYPE = "international"

# A match counts if its FIRST day is between these two dates (both days included).
# We use the first day so that a Test match that starts inside the window is never cut in half.
STUDY_START_DATE = date(2020, 1, 1)
STUDY_END_DATE = date(2026, 10, 29)

# Cricsheet writes "T20" (older files sometimes "IT20") for T20 internationals. We call it "T20I" everywhere.
CRICSHEET_TO_FORMAT = {"Test": "Test", "ODI": "ODI", "T20": "T20I", "IT20": "T20I"}
FORMATS = ("Test", "ODI", "T20I")
FORMAT_FILE_TAG = {"Test": "test", "ODI": "odi", "T20I": "t20i"}   # short names used in file names

## Settings that keep the results the same on every run

Louvain community detection and the random comparison graphs use random numbers. Fixing the seed means that
anyone who runs the notebooks gets exactly our numbers.

In [ ]:
RANDOM_SEED = 42
LOUVAIN_RESOLUTION = 1.0          # the standard modularity, we did not tune it
NUMBER_OF_RANDOM_GRAPHS = 200     # random graphs for the small-world comparison
LOUVAIN_STABILITY_RUNS = 100      # we also run Louvain with seeds 0 to 99 to see how much it moves

DATE_TEXT_FORMAT = "%d-%m-%Y"     # all dates in our CSV files are written as DD-MM-YYYY


def create_output_folders() -> None:
    """Make sure every output folder exists before a stage tries to write into it."""
    for folder in (INTERIM_FOLDER, PROCESSED_FOLDER, TABLES_FOLDER, FIGURES_FOLDER, LOGS_FOLDER):
        folder.mkdir(parents=True, exist_ok=True)


create_output_folders()

## Logging

Each stage prints what it is doing and also writes the same lines to `outputs/logs/<stage>.log`. This way we
always have a record of the last run, even after the notebook output is cleared.

In [ ]:
def get_stage_logger(stage_name: str) -> logging.Logger:
    """Return a logger that prints in the notebook and writes to outputs/logs/<stage_name>.log."""
    stage_logger = logging.getLogger(f"partnerships.{stage_name}")
    stage_logger.setLevel(logging.INFO)
    stage_logger.propagate = False

    # Edge case: if a cell is run twice, the logger would get a second set of handlers and print every
    # line twice. So we remove the old handlers first.
    for old_handler in list(stage_logger.handlers):
        stage_logger.removeHandler(old_handler)
        old_handler.close()

    line_format = logging.Formatter("%(asctime)s %(levelname)s %(name)s: %(message)s")
    file_handler = logging.FileHandler(LOGS_FOLDER / f"{stage_name}.log", mode="w", encoding="utf-8")
    screen_handler = logging.StreamHandler()
    for handler in (file_handler, screen_handler):
        handler.setFormatter(line_format)
        stage_logger.addHandler(handler)
    return stage_logger

## Saving and loading tables

All tables are UTF-8 CSV files. Dates are saved as `DD-MM-YYYY` and come back as real dates when we load them.

One thing we learned the hard way: pandas reads a column such as `match_id` (values like `1022347`) as a
number unless we tell it otherwise. IDs are labels, not numbers, so we always read ID columns as text.

In [ ]:
DATE_COLUMNS = ("match_date", "start_date", "end_date", "first_date", "last_date")
ID_COLUMNS = ("match_id", "player_id", "player_1_id", "player_2_id",
              "start_delivery_id", "end_delivery_id", "delivery_id")


def write_table(table: pd.DataFrame, output_path: Path) -> None:
    """Save a table as CSV, writing every date column as DD-MM-YYYY."""
    table_to_save = table.copy()
    for column_name in DATE_COLUMNS:
        if column_name in table_to_save.columns:
            table_to_save[column_name] = pd.to_datetime(table_to_save[column_name]).dt.strftime(DATE_TEXT_FORMAT)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    table_to_save.to_csv(output_path, index=False, encoding="utf-8")


def read_table(input_path: Path) -> pd.DataFrame:
    """Load a CSV saved with write_table. IDs stay text and dates become datetime.date values."""
    header_columns = pd.read_csv(input_path, nrows=0).columns
    text_columns = {column_name: str for column_name in ID_COLUMNS if column_name in header_columns}
    table = pd.read_csv(input_path, dtype=text_columns)
    for column_name in DATE_COLUMNS:
        if column_name in table.columns:
            table[column_name] = pd.to_datetime(table[column_name], format=DATE_TEXT_FORMAT).dt.date
    return table


def markdown_table(table: pd.DataFrame) -> str:
    """Turn a small table into Markdown so we can paste it into the data-quality report."""
    header_line = "| " + " | ".join(map(str, table.columns)) + " |"
    divider_line = "|" + "---|" * len(table.columns)
    body_lines = ["| " + " | ".join(map(str, row)) + " |" for row in table.itertuples(index=False)]
    return "\n".join([header_line, divider_line, *body_lines])